# YOLO Thermal Dataset Training

In [ ]:
%pip install pytz pandas 

In [ ]:
%load_ext autoreload
from logger import get_logger
logger = get_logger("YOLO-TRAIN")

## Environment

### Verificar CUDA en env

In [ ]:
import torch

torch.cuda.is_available()

### Verificar librería ultralytics

In [ ]:
# Issue por compatibilidad !!! 8.3.80
%pip install -qU ultralytics==8.3.80

In [ ]:
from ultralytics import __version__ as ultralytics_version

ultralytics_version

### Importar dependencias

Ingresamos la ruta del dataset

In [ ]:
import os
import pandas as pd
from datetime import datetime
from ultralytics import YOLO

# DATASET_PATH = os.path.join(os.getcwd(), "data/desmodus_thermal.v1i.yolov11")
DATASET_PATH = os.path.join(os.getcwd(), "data/DESMUDIN.v1i.yolov11")

## YOLO series

Definimos los métodos de entrenamiento y exportación a formatos PT (PYTORCH) y TFLITE

Los hiperparámetros definidos son: 
device="cuda",
imgsz=640,
batch=16,
workers=64,
epochs=50,
pretrained=False,

In [ ]:
def train_yolo_model(model: YOLO, seed: int = 0):
    """Entrena el modelo yolo con el dataset de lissachatina"""
    res = model.train(
        data=os.path.join(DATASET_PATH, "data.yaml"),
        # optimizer="auto", # SGD, Adam, AdamW, NAdam, RAdam, RMSProp etc., or auto
        # lr0=0.01, #  (i.e. SGD=1E-2, Adam=1E-3)
        seed=seed,
        close_mosaic=True,
        device="cuda",
        imgsz=640,
        batch=16,
        # workers=64,
        workers=0,
        epochs=100,
        pretrained=False,
        patience=15,
    )

    return res


def export_yolo_model(model: YOLO) -> str:
    """Exporta el modelo yolo a tflite con Float16"""

    res_dir = model.export(
        format="tflite",
        half=True,
        # int8=True,
        imgsz=320,
        workers=0,
        # workers=64,
        device="cuda",
        data=os.path.join(DATASET_PATH, "data.yaml"),
    )

    return res_dir


def save_results_to_csv(trained_yolo_path: dict[tuple, str], name: str):
    """Guarda resultados de modelo, semilla y path en un csv"""
    df = pd.DataFrame(
        [
            {"yolo": yolo, "seed": seed, "path": path, "dataset": DATASET_PATH}
            for (yolo, seed), path in trained_yolo_path.items()
        ]
    )

    # Save to CSV
    name = name if name.endswith(".csv") else f"{name}.csv"
    df.to_csv(name, index=False)

### Train YOLO's (.pt)

In [ ]:
trained_yolo_paths: dict[tuple, str] = {}

# Train for 2 different seeds
for seed in [3000]:
    for yolo in ["yolov8n", "yolov9t", "yolov10n", "yolo11n", "yolo12n"]:
        logger.info(f"Entrenando YOLO {yolo} con seed {seed}")

        yolo_model = YOLO(yolo)
        results = train_yolo_model(model=yolo_model, seed=seed)
        best_model_path = f"{str(results.save_dir)}/weights/best.pt"

        trained_yolo_paths[(yolo, seed)] = best_model_path
        logger.info("Guardado en %s", best_model_path)

TIMESTAMP = datetime.now().isoformat().replace(":", "-").replace(".", "-")
filename = f"{TIMESTAMP}_entrenamiento_yolo"
save_results_to_csv(trained_yolo_paths, filename)

logger.info("CSV file '%s' saved successfully.", filename)
trained_yolo_paths

### Export YOLO's to .tflite

In [ ]:
# exported_yolo_paths = {}

# for (model, seed), model_path in trained_yolo_paths.items():
#     logger.info("### Exportando modelo %s con seed %s desde %s...", model, seed, model_path)

#     yolo_model = YOLO(model_path)
#     res_dir = export_yolo_model(model=yolo_model)

#     exported_yolo_paths[(model, seed)] = res_dir
#     logger.info("Exportado en %s", res_dir)

# # Save to CSV
# TIMESTAMP = datetime.now().isoformat()
# filename = f"{TIMESTAMP}_exportado_yolo"
# save_results_to_csv(exported_yolo_paths, filename)

# logger.info("CSV file '%s' saved successfully.", filename)
# exported_yolo_paths